# 9. Build CPD + N4T pairs at a chosen geometry
**Input:** nothing required (RDKit templates), or a trajectory frame / your own relaxed molecules.
**Output:** finite pair structures at requested distance, ring dihedral and plane tilt, with the measured metrics; optional `pairs.extxyz`, table and configuration.
**Modules:** `chemistry.cpd_n4t.build` (placement), `chemistry.cpd_n4t.selection` (measurement, the same equations as the trajectory analysis).

The N4T (dienophile) is fixed; the CPD is moved as a rigid body. These are geometric starting structures for reaction-method work, not optimized structures, transition states or energies.

In [ ]:
from pathlib import Path
import json
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ase.io import read, write
from MartiniFuncs.chemistry.cpd_n4t.build import build_pair, rdkit_templates, templates_from_frame
from MartiniFuncs.chemistry.cpd_n4t.selection import measure_pair_geometry

# Templates: "rdkit" (built from SMILES, MMFF-relaxed), "frame" (first whole CPD and
# N4T in TEMPLATE_FRAME_FILE) or "files" (your own relaxed CPD_FILE and N4T_FILE).
TEMPLATE_SOURCE = "rdkit"
RDKIT_SEED = 61453
TEMPLATE_FRAME_FILE = Path(r"/path/to/trajectory.lammpstrj")   # TEMPLATE_SOURCE="frame"
TEMPLATE_FRAME_INDEX = 0
UNITS = "metal"
CPD_FILE = Path(r"/path/to/cpd.xyz")                            # TEMPLATE_SOURCE="files"
N4T_FILE = Path(r"/path/to/n4t.xyz")

# One structure is built for every combination of these values.
DISTANCES = [2.6, 3.0, 3.5]          # Angstrom; meaning set by DISTANCE_KIND
DISTANCE_KIND = "centers"            # "centers", "max_forming" or "screen"
RING_DIHEDRALS = [0.0, 180.0]        # degrees; near 0 exo, near +/-180 endo
PLANE_ANGLES = [0.0, 15.0]           # degrees, -90 < angle < 90; + opens the CH2 side away
FACE = 1                             # +1 or -1: which N4T face the CPD approaches
MIN_CONTACT = 1.0                    # Angstrom; skip structures with closer intermolecular atoms (None: keep all)

SHOW_3D = False                      # Psience view of the first structure (needs Psience/McUtils)
OUTPUT_DIR = Path(r"/path/to/built_pairs")
WRITE_FILES = False

### Mathematics: points, dihedral and placement
Sites come from the same identification as the trajectory analysis: $P_1$ is the CPD CH$_2$ carbon, $P_2$ the midpoint of the two diene termini (the CPD carbons that bond to the dienophile), $P_3$ the midpoint of the N4T C=C, and $P_4$ the imide N (bonded to both carbonyl carbons).

The ring dihedral uses $\mathbf b_1=P_2-P_1$, $\mathbf b_2=P_3-P_2$, $\mathbf b_3=P_4-P_3$:
$$\varphi=\operatorname{atan2}\!\big(|\mathbf b_2|\,\mathbf b_1\cdot(\mathbf b_2\times\mathbf b_3),\;(\mathbf b_1\times\mathbf b_2)\cdot(\mathbf b_2\times\mathbf b_3)\big),$$
in $(-180^\circ,180^\circ]$. $\varphi\approx0$: the imide ring points the same way as the CH$_2$ bridge (exo). $|\varphi|\approx180^\circ$: it lies under the diene (endo). The sign is the twist direction (mirror images).

Placement, in the fixed N4T frame: the approach axis $\hat{\mathbf u}$ is perpendicular to $P_4-P_3$ and to the C=C, on side `FACE`. The CPD is rotated so $P_1-P_2$ is $P_4-P_3$ turned by $-\varphi$ about $\hat{\mathbf u}$ with its ring perpendicular to $\hat{\mathbf u}$, then tilted by the plane angle $\theta$ about the axis through $P_2$ perpendicular to $\hat{\mathbf u}$ and $P_1-P_2$; finally $P_2=P_3-d\,\hat{\mathbf u}$. The tilt does not change $\varphi$.

`DISTANCE_KIND` sets what $d$ controls: `centers` $|P_3-P_2|=d$; `max_forming` the longer forming C$\cdots$C distance (the `max_forming_distance` metric); `screen` the whole-CPD COM to C=C COM distance (the analysis `reactive_radius` screen). $\varphi$ and the distance are met to numerical precision; the measured `plane_angle` and `approach_angle` equal $|\theta|$ up to the templates' non-planarity. All distances are Å, angles degrees.

In [ ]:
if TEMPLATE_SOURCE == "rdkit":
    cpd, n4t = rdkit_templates(seed=RDKIT_SEED)
elif TEMPLATE_SOURCE == "frame":
    from MartiniFuncs.io.lammps import read_lammps_frame
    cpd, n4t = templates_from_frame(read_lammps_frame(TEMPLATE_FRAME_FILE, index=TEMPLATE_FRAME_INDEX, units=UNITS).atoms)
elif TEMPLATE_SOURCE == "files":
    cpd, n4t = read(CPD_FILE), read(N4T_FILE)
else:
    raise ValueError('TEMPLATE_SOURCE must be "rdkit", "frame" or "files"')
print("CPD template:", cpd.get_chemical_formula(), f"({len(cpd)} atoms);",
      "N4T template:", n4t.get_chemical_formula(), f"({len(n4t)} atoms)")

## One structure, checked two ways
The builder measures its result directly. As an independent check, the next cell also rebuilds the bond graph of the combined pair and measures it exactly as a trajectory frame would be measured.

In [ ]:
first = dict(distance=DISTANCES[0], ring_dihedral=RING_DIHEDRALS[0], plane_angle=PLANE_ANGLES[0])
example, geometry = build_pair(cpd, n4t, distance_kind=DISTANCE_KIND, face=FACE, **first)
ids, roles = example.arrays["global_index"], example.arrays["pair_role"]
check = measure_pair_geometry(example, [ids[roles == "CPD"], ids[roles == "MAL"]], (0, 0, 0))


def distances(g):
    """Center separation, longer forming distance and screen distance (CPD COM to C=C COM), Angstrom."""
    termini, cc = [b[0] for b in g.forming_bonds], [b[1] for b in g.forming_bonds]
    cpd_rows = np.flatnonzero(roles == "CPD")
    return [np.linalg.norm(example.positions[cc].mean(0) - example.positions[termini].mean(0)),
            max(g.forming_distances),
            np.linalg.norm(example[cpd_rows].get_center_of_mass() - example[cc].get_center_of_mass())]


names = ["center separation", "max_forming_distance", "screen distance",
         "ring_dihedral", "plane_angle", "axis_angle", "approach_angle"]
requested = [None, None, None, first["ring_dihedral"], abs(first["plane_angle"]), None, None]
requested[{"centers": 0, "max_forming": 1, "screen": 2}[DISTANCE_KIND]] = first["distance"]
display(pd.DataFrame({
    "requested": requested,
    "measured (builder)": distances(geometry) + [geometry.ring_dihedral, geometry.plane_angle,
                                                 geometry.axis_angle, geometry.approach_angle],
    "measured (rebuilt graph)": distances(check) + [check.ring_dihedral, check.plane_angle,
                                                    check.axis_angle, check.approach_angle],
}, index=names))
print("Approach:", geometry.approach_type, "| closest intermolecular atoms:",
      f"{example.info['min_intermolecular_distance']:.3f} Angstrom")

## Scan the requested grid
Every combination of `DISTANCES`, `RING_DIHEDRALS` and `PLANE_ANGLES` is built. Structures closer than `MIN_CONTACT` (or impossible requests) are listed with the reason instead of being kept.

In [ ]:
structures, rows = [], []
for distance, dihedral, tilt in itertools.product(DISTANCES, RING_DIHEDRALS, PLANE_ANGLES):
    row = dict(requested_distance=distance, distance_kind=DISTANCE_KIND,
               requested_ring_dihedral=dihedral, requested_plane_angle=tilt)
    try:
        atoms, g = build_pair(cpd, n4t, distance=distance, ring_dihedral=dihedral, plane_angle=tilt,
                              distance_kind=DISTANCE_KIND, face=FACE, min_contact=MIN_CONTACT)
    except ValueError as exc:
        rows.append(dict(row, kept=False, reason=str(exc)))
        continue
    atoms.info["structure_index"] = len(structures)
    structures.append(atoms)
    rows.append(dict(row, kept=True, reason="", structure_index=atoms.info["structure_index"],
                     center_separation=atoms.info["center_separation"],
                     max_forming_distance=max(g.forming_distances), distance_imbalance=g.distance_imbalance,
                     ring_dihedral=g.ring_dihedral, approach_type=g.approach_type, plane_angle=g.plane_angle,
                     axis_angle=g.axis_angle, approach_angle=g.approach_angle,
                     min_intermolecular_distance=atoms.info["min_intermolecular_distance"]))
table = pd.DataFrame(rows)
print(f"Built {len(structures)} of {len(table)} requested structures")
display(table)

In [ ]:
kept = table[table["kept"]]
if len(kept):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), layout="constrained")
    wrapped = (kept["requested_ring_dihedral"] + 180) % 360 - 180
    axes[0].scatter(wrapped, kept["ring_dihedral"])
    axes[0].set(xlabel="requested ring dihedral (deg)", ylabel="measured (deg)", title="Ring dihedral")
    axes[1].scatter(kept["requested_plane_angle"].abs(), kept["plane_angle"])
    axes[1].set(xlabel="|requested plane angle| (deg)", ylabel="measured (deg)", title="Plane angle")
    for name, group in kept.groupby("requested_ring_dihedral"):
        axes[2].plot(group["requested_distance"], group["min_intermolecular_distance"], "o", label=f"phi={name:g}")
    axes[2].set(xlabel=f"requested distance ({DISTANCE_KIND}, Angstrom)", ylabel="closest contact (Angstrom)",
                title="Closest intermolecular atoms")
    axes[2].legend()
    for ax in axes[:2]:
        lo, hi = ax.get_xlim()
        ax.plot([lo, hi], [lo, hi], color="gray", linewidth=.8)
    plt.show()

In [ ]:
if SHOW_3D and structures:
    from MartiniFuncs.chemistry.cpd_n4t.visualization import plot_pair_geometry
    display(plot_pair_geometry(structures[0]))

## Save
With `WRITE_FILES=True`, a new `OUTPUT_DIR` receives `pairs.extxyz` (one frame per kept structure, with `global_index`, `pair_role` and the requested and measured values in each frame's info), `pairs.csv` (the full table, including skipped requests) and `configuration.json`. Existing output is never overwritten.

In [ ]:
if WRITE_FILES:
    if OUTPUT_DIR.exists():
        raise FileExistsError(f"Choose a new OUTPUT_DIR: {OUTPUT_DIR}")
    OUTPUT_DIR.mkdir(parents=True)
    write(OUTPUT_DIR / "pairs.extxyz", structures, format="extxyz")
    table.to_csv(OUTPUT_DIR / "pairs.csv", index=False)
    (OUTPUT_DIR / "configuration.json").write_text(json.dumps(dict(
        template_source=TEMPLATE_SOURCE, rdkit_seed=RDKIT_SEED if TEMPLATE_SOURCE == "rdkit" else None,
        template_frame_file=str(TEMPLATE_FRAME_FILE) if TEMPLATE_SOURCE == "frame" else None,
        template_frame_index=TEMPLATE_FRAME_INDEX if TEMPLATE_SOURCE == "frame" else None,
        cpd_file=str(CPD_FILE) if TEMPLATE_SOURCE == "files" else None,
        n4t_file=str(N4T_FILE) if TEMPLATE_SOURCE == "files" else None,
        distances=DISTANCES, distance_kind=DISTANCE_KIND, ring_dihedrals=RING_DIHEDRALS,
        plane_angles=PLANE_ANGLES, face=FACE, min_contact=MIN_CONTACT), indent=2) + "\n")
    print("Wrote", OUTPUT_DIR)